# Lilly Listen — clean shipped-model evaluation

Eval only: exact shipped `lilly-listen-large-v3` plus the registered small
baseline, 925 hash-pinned FLEURS Bosnian test recordings, product decode path,
WER by frozen legibility cohort, and the two registered language gates. No
training, install, product-default change, publication, or leaderboard upload.


In [ ]:
JOB = "listen-clean-eval"
import hashlib, json, os, shutil, subprocess, sys, urllib.error, urllib.request, zipfile
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
try:
    GPU = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).splitlines()[0].strip()
except (FileNotFoundError, IndexError, subprocess.CalledProcessError) as exc:
    raise SystemExit(f"Kaggle GPU is required for this heavy evaluation: {exc}")
print("using GPU", GPU)

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(f"network unavailable for {url}: {exc}")
for host in ("https://github.com", "https://pypi.org", "https://huggingface.co"):
    reachable(host)

TEE = Path("/kaggle/working/stdout.txt")
# Offload writes /kaggle/working/experiment_log.json beside the evidence zip.
TEE.parent.mkdir(parents=True, exist_ok=True)
def run(*cmd, quiet=False, env=None):
    line = "$ " + " ".join(str(x) for x in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1,
                                 env={**os.environ, **(env or {})})
        for output in child.stdout:
            if not quiet:
                print(output, end="", flush=True)
            sink.write(output)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)


In [ ]:
EXPECTED_GIT_COMMIT = "__LAUNCHER_GIT_COMMIT__"
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
if CLONE.exists():
    shutil.rmtree(CLONE)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git", str(CLONE))
os.chdir(CLONE)
run("git", "checkout", "-q", EXPECTED_GIT_COMMIT)
got_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if got_commit != EXPECTED_GIT_COMMIT:
    raise SystemExit(f"clone is {got_commit}, expected {EXPECTED_GIT_COMMIT}")
sys.path.insert(0, str(CLONE))
from training.kaggle_offload import Offload
OFF = Offload(JOB, os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "manual"))
OFF.hardware(GPU)
print("exact code commit", got_commit)


In [ ]:
packages = ["faster-whisper==1.2.1", "ctranslate2==4.8.1", "soundfile==0.14.0",
            "pyarrow==25.0.1", "huggingface-hub==0.36.0", "transformers==4.49.0"]
run(sys.executable, "-m", "pip", "install", "-q", *packages)
import ctranslate2, faster_whisper, soundfile
print("runtime", faster_whisper.__version__, ctranslate2.__version__, soundfile.__version__)
print("CUDA compute types", ctranslate2.get_supported_compute_types("cuda"))


In [ ]:
SOURCE = CLONE / "training/clean-eval/speech-source.json"
MANIFEST = CLONE / "training/clean-eval/speech-fleurs-bs-test.tsv"
source = json.loads(SOURCE.read_text(encoding="utf-8"))
if hashlib.sha256(MANIFEST.read_bytes()).hexdigest() != source["manifest_sha256"]:
    raise SystemExit("committed speech manifest does not match source record")
run(sys.executable, "training/fetch_pinned_fleurs_test.py", "--source", str(SOURCE),
    "--manifest", str(MANIFEST), "--out", str(CLONE / "data/speech"))
rows = (CLONE / "data/speech/test.tsv").read_text(encoding="utf-8").splitlines()
if len(rows) != 925:
    raise SystemExit(f"FLEURS test has {len(rows)} rows, not 925")
run(sys.executable, "training/probe_speech_clips.py", "--data", "data/speech/test.tsv",
    "--out", "/kaggle/working/speech-waveform-diagnostics.json")
OFF.metric("speech_clips", 925, stage="data")
OFF.metric("manifest_sha256", source["manifest_sha256"], stage="data")


In [ ]:
PINS = {"listen-previous": "a76342f6ab59b382", "listen": "e6bb58483586b06c"}
BASES = {"listen-previous": "openai/whisper-small", "listen": "openai/whisper-large-v3"}
def listener_fingerprint(build):
    h = hashlib.md5()
    for name in sorted(p.name for p in build.iterdir()
                       if p.is_file() and p.name != "dataset-metadata.json"):
        h.update(name.encode()); h.update((build / name).read_bytes())
    return h.hexdigest()[:16]
attached = [p.parent for p in Path("/kaggle/input").rglob("built.json")
            if (p.parent / "model.bin").is_file()]
for label in ("listen-previous", "listen"):
    matches = [p for p in attached
               if json.loads((p / "built.json").read_text()).get("base") == BASES[label]]
    if len(matches) != 1:
        raise SystemExit(f"{label}: need exactly one attached {BASES[label]}, found {matches}")
    destination = CLONE / "models/lilly" / label
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(matches[0], destination,
                    ignore=shutil.ignore_patterns("dataset-metadata.json"))
    got = listener_fingerprint(destination)
    if got != PINS[label]:
        raise SystemExit(f"{label} fingerprint {got}, expected {PINS[label]}")
    print(label, json.loads((destination / "built.json").read_text()), got,
          (destination / "model.bin").stat().st_size)


In [ ]:
GPU_ENV = {"LILLY_SPEECH_DEVICE": "cuda", "CT2_CUDA_ALLOW_FP16": "1"}
PREV = CLONE / "models/lilly/listen-previous"
SHIPPED = CLONE / "models/lilly/listen"
run(sys.executable, "training/speech_bench.py", "--clips", "all", "--limit", "8",
    "--model", str(PREV), "--model", str(SHIPPED),
    "--json", "/kaggle/working/speech-smoke.json", env=GPU_ENV)
GATE = Path("/kaggle/working/speech-gate.json")
run(sys.executable, "training/speech_bench.py", "--clips", "all",
    "--model", str(PREV), "--model", str(SHIPPED), "--json", str(GATE), env=GPU_ENV)
gate = json.loads(GATE.read_text())
if gate["n_clips"] != 925 or gate["n_clips_in_split"] != 925:
    raise SystemExit(f"speech gate scored partial data: {gate['n_clips']}/{gate['n_clips_in_split']}")
WER = Path("/kaggle/working/speech-legibility.json")
run(sys.executable, "training/speech_legibility_report.py", "--data", "data/speech/test.tsv",
    "--manifest", str(MANIFEST), "--cache", "bench/speech/.outputs.json",
    "--model", str(PREV), "--model", str(SHIPPED), "--json", str(WER),
    "--markdown", "/kaggle/working/speech-legibility.md")
# Supplementary only: pin the official Open ASR code, apply its multilingual
# normalizer/scoring to the same cached predictions, and upload nothing.
OPEN_ASR_COMMIT = "4ef4a26b8588ccc140868be36f8c34acc839afe6"
OPEN_ASR = SCRATCH / "open_asr_leaderboard"
if OPEN_ASR.exists():
    shutil.rmtree(OPEN_ASR)
run("git", "clone", "-q", "https://github.com/huggingface/open_asr_leaderboard.git", str(OPEN_ASR))
run("git", "-C", str(OPEN_ASR), "checkout", "-q", OPEN_ASR_COMMIT)
run(sys.executable, "-m", "pip", "install", "-q", "num2words==0.5.14", "kaldialign==0.9.1")
OPEN_ASR_JSON = Path("/kaggle/working/open-asr-offline.json")
run(sys.executable, "training/open_asr_offline_score.py", "--predictions", str(WER),
    "--harness", str(OPEN_ASR), "--harness-commit", OPEN_ASR_COMMIT,
    "--json", str(OPEN_ASR_JSON))


In [ ]:
prev = gate["listeners"]["listen-previous"]
shipped = gate["listeners"]["listen"]
checks = {
    "term_recall_not_below_baseline": shipped["term_recall"] >= prev["term_recall"],
    "croatian_substitution_not_above_baseline": shipped["croatian"] <= prev["croatian"],
}
verdict = "PASS" if all(checks.values()) else "FAIL"
decision = {"schema": 1, "artifact": "shipped lilly-listen-large-v3",
            "fingerprints": PINS, "registered_gates": checks, "verdict": verdict,
            "baseline": prev, "shipped": shipped}
DECISION = Path("/kaggle/working/speech-clean-eval-decision.json")
DECISION.write_text(json.dumps(decision, ensure_ascii=False, indent=2) + "\n")
print("registered gate verdict", verdict, checks)
OFF.check_trainproof(TEE)
files = [GATE, WER, DECISION, OPEN_ASR_JSON, Path("/kaggle/working/speech-legibility.md"),
         Path("/kaggle/working/speech-waveform-diagnostics.json"),
         CLONE / "bench/speech/.outputs.json", MANIFEST, SOURCE]
for path in files:
    if not path.is_file() or path.stat().st_size == 0:
        raise SystemExit(f"missing result {path}")
ZIP = Path("/kaggle/working/lilly-listen-clean-eval.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.name)
if ZIP.stat().st_size < 20_000:
    raise SystemExit(f"result zip too small: {ZIP.stat().st_size}")
OFF.finish("complete", [ZIP.name])
print("wrote", ZIP, ZIP.stat().st_size, "bytes; eval only, no weights or defaults changed")
